# 04 - Stockout Risk (LeakLens)
**Goal:** Estimate the risk that a product will stock out soon.

**Approach:** compute recent sales velocity, define a clear high-risk label, then train ONE LogisticRegression to produce a smooth risk probability.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

DATA = Path("../data")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

def first_existing(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None

In [2]:
inventory = pd.read_csv(DATA / "inventory.csv")
products = pd.read_csv(DATA / "products.csv")
transactions = pd.read_csv(DATA / "transactions.csv")

print("inventory    :", inventory.shape)
print("products     :", products.shape)
print("transactions :", transactions.shape)

inventory    : (5000, 7)
products     : (250, 5)
transactions : (18000, 10)


In [3]:
date_col = first_existing(transactions, ["transaction_date", "date", "order_date", "timestamp"])
qty_col = first_existing(transactions, ["quantity", "qty", "units"])
assert date_col is not None, "No date column found in transactions"
assert "product_id" in transactions.columns, "transactions needs product_id"

transactions[date_col] = pd.to_datetime(transactions[date_col], errors="coerce")
WINDOW_DAYS = 30
cutoff = transactions[date_col].max() - pd.Timedelta(days=WINDOW_DAYS)
recent = transactions[transactions[date_col] > cutoff]

if qty_col:
    units = recent.groupby("product_id")[qty_col].sum()
else:
    units = recent.groupby("product_id").size()   # fallback: count of sales rows

velocity = (units / WINDOW_DAYS).rename("velocity").reset_index()   # units per day
print(f"Velocity window: last {WINDOW_DAYS} days up to {transactions[date_col].max().date()}")
print("Products with velocity:", velocity.shape[0])
velocity["velocity"].describe()

Velocity window: last 30 days up to 2025-12-31
Products with velocity: 250


count    250.000000
mean       0.580267
std        0.271035
min        0.033333
25%        0.366667
50%        0.566667
75%        0.758333
max        1.433333
Name: velocity, dtype: float64

In [4]:
assert "product_id" in inventory.columns, "inventory needs product_id"

df = inventory.copy()
prod_cols = ["product_id"] + [c for c in ["category"] if c in products.columns and c not in df.columns]
df = df.merge(products[prod_cols].drop_duplicates("product_id"), on="product_id", how="left")
df = df.merge(velocity, on="product_id", how="left")
df["velocity"] = df["velocity"].fillna(0)

print("Joined shape:", df.shape)
df.head()

Joined shape: (5000, 9)


,inventory_record_id,product_id,date,stock_on_hand,reorder_threshold,stockout_flag,supplier_delay_days,category,velocity
0,IN000001,PR0160,2025-03-16,221,23,0,18,Electronics,0.233333
1,IN000002,PR0076,2025-03-07,142,65,0,9,Electronics,0.333333
2,IN000003,PR0113,2025-11-02,203,38,0,11,Home,0.733333
3,IN000004,PR0206,2025-07-13,195,12,0,15,Electronics,0.266667
4,IN000005,PR0244,2025-09-29,178,50,0,1,Electronics,0.333333


In [5]:
assert "stock_on_hand" in df.columns and "reorder_threshold" in df.columns, \
    "Need stock_on_hand and reorder_threshold to define the target"

velocity_median = df["velocity"].median()
low_stock = df["stock_on_hand"] <= df["reorder_threshold"]
fast_seller = df["velocity"] >= velocity_median

# HIGH RISK = stock at/below reorder threshold AND selling at/above median speed
df["high_risk"] = (low_stock & fast_seller).astype(int)

print("Target definition: stock_on_hand <= reorder_threshold AND velocity >= median velocity")
print(f"Median velocity: {velocity_median:.3f} units/day")
print("Class balance:")
print(df["high_risk"].value_counts().rename({0: "not high risk", 1: "high risk"}))
assert df["high_risk"].nunique() == 2, "Target has only one class - adjust the definition"

Target definition: stock_on_hand <= reorder_threshold AND velocity >= median velocity
Median velocity: 0.567 units/day
Class balance:
high_risk
not high risk    4624
high risk         376
Name: count, dtype: int64


In [6]:
numeric_candidates = ["stock_on_hand", "reorder_threshold", "supplier_lead_days",
                      "supplier_delay_days", "velocity"]
numeric_features = [c for c in numeric_candidates if c in df.columns]
categorical_features = [c for c in ["category"] if c in df.columns]

numeric_fill = df[numeric_features].median().to_dict()
df[numeric_features] = df[numeric_features].fillna(numeric_fill)
if categorical_features:
    df[categorical_features] = df[categorical_features].fillna("unknown")

X = df[numeric_features + categorical_features]
y = df["high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Numeric features    :", numeric_features)
print("Categorical features:", categorical_features)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

Numeric features    : ['stock_on_hand', 'reorder_threshold', 'supplier_delay_days', 'velocity']
Categorical features: ['category']
Train shape: (4000, 5) | Test shape: (1000, 5)


In [7]:
transformers = [("num", StandardScaler(), numeric_features)]
if categorical_features:
    transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features))

pipe = Pipeline([
    ("prep", ColumnTransformer(transformers)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
pipe.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [8]:
proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"Test AUC: {roc_auc_score(y_test, proba):.3f}\n")
print(classification_report(y_test, pred, target_names=["not high risk", "high risk"]))

# Explainability: strongest coefficients
names = pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(pipe.named_steps["clf"].coef_[0], index=names).sort_values(key=abs, ascending=False)
print("Top drivers of stockout risk (coefficient):")
print(coefs.head(8).round(3))

# Risk bands on the whole table
df["risk_score"] = pipe.predict_proba(X)[:, 1]
df["risk_band"] = pd.cut(df["risk_score"], [-0.01, 0.33, 0.66, 1.0], labels=["Low", "Medium", "High"])
print("\nRisk band counts:")
print(df["risk_band"].value_counts())

show = [c for c in ["product_id", "category", "stock_on_hand", "reorder_threshold", "velocity", "risk_score"] if c in df.columns]
print("\nTop 10 at-risk inventory rows:")
display(df.sort_values("risk_score", ascending=False)[show].head(10))

Test AUC: 0.984

               precision    recall  f1-score   support

not high risk       1.00      0.93      0.97       925
    high risk       0.55      0.99      0.70        75

     accuracy                           0.94      1000
    macro avg       0.77      0.96      0.84      1000
 weighted avg       0.97      0.94      0.95      1000

Top drivers of stockout risk (coefficient):
num__stock_on_hand          -8.263
num__velocity                2.506
num__reorder_threshold       1.654
cat__category_Electronics   -0.509
cat__category_Sports        -0.413
cat__category_Grocery        0.341
cat__category_Home           0.310
cat__category_Fashion        0.189
dtype: float64

Risk band counts:
risk_band
Low       4246
High       579
Medium     175
Name: count, dtype: int64

Top 10 at-risk inventory rows:


,product_id,category,stock_on_hand,reorder_threshold,velocity,risk_score
2571,PR0073,Sports,5,63,1.433333,0.999995
3236,PR0102,Sports,7,61,1.333333,0.999984
951,PR0222,Fashion,8,53,1.266667,0.999968
245,PR0102,Sports,27,71,1.333333,0.999954
2391,PR0108,Electronics,9,52,1.333333,0.999952
1849,PR0222,Fashion,24,68,1.266667,0.999949
3961,PR0161,Home,13,76,1.033333,0.999946
2603,PR0102,Sports,33,75,1.333333,0.999941
3811,PR0039,Grocery,2,55,1.100000,0.999936
4115,PR0184,Electronics,5,71,1.066667,0.999934


In [9]:
artifact = {
    "model": pipe,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "numeric_fill": numeric_fill,
    "velocity_window_days": WINDOW_DAYS,
    "velocity_median": float(velocity_median),
    "target_definition": "stock_on_hand <= reorder_threshold AND velocity >= median velocity",
}
joblib.dump(artifact, MODELS / "stockout_model.joblib")
print("Saved:", MODELS / "stockout_model.joblib")

Saved: ..\models\stockout_model.joblib


## Summary for judges
- **Clear label:** "high risk" means stock is at or below the reorder threshold while the product sells faster than the median, a rule any operations manager would agree with.
- **Explainable model:** one LogisticRegression gives a 0-1 risk score, and its coefficients show which factors (stock level, velocity, supplier lead time) push risk up.
- **Honest caveat:** because the label comes from a business rule, AUC shows how well the model reproduces that rule. In production, the label would be replaced with real past stockout events.
- **Business value:** products can be ranked by risk so reorders happen before lost sales occur.